# Fine Tuning Assignment (Graded): Customer Service Response Generator using Few-Shot Learning

Welcome to your programming assignment on prompting! You will build a comprehensive Customer Service Response Generator.

> **Modifications in this version:** the bot now uses `gpt-6-luna` instead of the retired `gpt-3.5-turbo`; the API key loads from a `.env` file instead of being pasted into the notebook; two tones (`reassuring`, `concise`) and two categories (`billing`, `order_status`) were added; and each category has its own response guidelines in the prompt.

## Problem Description

- Customer service teams often need to maintain consistency in their responses while personalizing them for each customer. 

- In this assignment, you will build a system that learns from examples of high-quality customer service interactions to generate appropriate responses for new customer inquiries.

## Assignment Tasks

**1. Class Setup and API Integration**
- Implement the `CustomerServiceBot` class constructor with OpenAI client integration


**2. Basic Prompting Implementation**
- Implement `generate_response` method with error handling to fetch response from OpenAI API


**3. Content Generation Methods**
- Implement `_create_prompt`.


**4. Few-Shot Learning Implementation**

## Instructions

- Only write code when you see any of the below prompts,

    ```
    # YOUR CODE GOES HERE
    # YOUR CODE ENDS HERE
    # TODO
    ```

- Do not modify any other section of the code unless tated otherwise in the comments.

# Code Section

In [ ]:
from openai import OpenAI
from typing import List, Dict, Optional
import os
import json
import logging
from pathlib import Path
from tests.test_methods import TestAICustomerServiceBot, print_test_results

## Task: Load your OpenAI API Key and choose the model

In [ ]:
# Load the API key from a .env file instead of hard-coding it.
# (A key pasted into a notebook gets exposed as soon as the repo is pushed.)
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not found. Create a .env file containing: OPENAI_API_KEY=your-key")

# --- Model configuration (modified from the original lab) ---
MODEL_NAME = "gpt-6-luna"     # replaces the retired gpt-3.5-turbo
REASONING_EFFORT = "none"     # fastest and cheapest for short replies
MAX_COMPLETION_TOKENS = 400

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Task: Implement the CustomerServiceBot Class

**Task Hints:**

- Implement the `__init__` method to initialize the bot fundamentals and configuration. You'll need to set up the OpenAI API client and prepare your working environment. 

- Implement the `load_examples` method to manage your example database. This method serves as the foundation for few-shot learning, so carefully consider how to organize and retrieve relevant examples.

- Implement the `_create_prompt` method to generate effective prompts for the AI. 

- Implement the `generate_response` method as your main interface for getting AI responses. 

In [ ]:
class CustomerServiceBot:
    """
    A customer service bot that uses few-shot learning to generate
    appropriate responses to customer inquiries.
    """
    
    def __init__(self, api_key: str, model: str = MODEL_NAME):
        """
        Initialize the customer service bot with OpenAI credentials
        
        Args:
            api_key: OpenAI API key
            model: OpenAI model to use for generating responses
        """
        # Initialize OpenAI client
        self.client = OpenAI(api_key=api_key)
        self.model = model
        
        # Define standard tones and their characteristics
        self.valid_tones = {
            "professional": "formal and business-like",
            "friendly": "warm and conversational",
            "formal": "highly polite and structured",
            "empathetic": "understanding and compassionate",
            "reassuring": "calm, confident, and focused on putting the customer at ease",
            "concise": "brief and direct, with no unnecessary detail"
        }
        
        # What a good answer should contain for each category
        self.category_guidelines = {
            "refund": "Start with a sincere apology (for example, 'Please accept our apology'), "
                      "confirm you will help with the refund, and explain the refund process and timeline.",
            "technical_support": "Introduce the instructions with 'Here are the steps:' and give clear "
                                 "numbered steps that solve the customer's specific problem.",
            "general_inquiry": "Answer the question directly, then offer further help.",
            "billing": "Acknowledge the billing concern, explain what you will check, and ask for the "
                       "details needed to look into it (such as the order or invoice number).",
            "order_status": "Explain how the customer can track the order and what details you need "
                            "to look it up."
        }
        
        # Load example database
        self.example_database = self._initialize_example_database()

    def _initialize_example_database(self) -> Dict[str, List[Dict]]:
        """Initialize the example database with predefined examples"""
        return {
            "refund": [
                {
                    "customer": "My product arrived damaged and I want a refund.",
                    "response": "I sincerely apologize for the damaged product. I understand this is frustrating. I'll be happy to help process your refund right away. Could you please provide your order number and a photo of the damage? Once received, we'll process your refund within 2-3 business days.",
                    "tone": "empathetic"
                },
                {
                    "customer": "I want to return my purchase for a refund.",
                    "response": "I'll assist you with your refund request. Please provide your order number and the reason for the return. Our refund policy allows returns within 30 days of purchase, and I'll guide you through the process step by step.",
                    "tone": "professional"
                }
            ],
            "technical_support": [
                {
                    "customer": "The app keeps crashing on my phone.",
                    "response": "I understand how frustrating app crashes can be. Let's solve this together. First, could you tell me your phone model and operating system version? Also, have you tried these quick fixes:\n1. Force-stopping the app\n2. Clearing the app cache\n3. Restarting your phone",
                    "tone": "friendly"
                },
                {
                    "customer": "I can't log into my account.",
                    "response": "Thank you for reporting this login issue. To assist you effectively, please confirm:\n1. Are you receiving any specific error messages?\n2. Have you tried resetting your password?\n3. Are you using the correct email address for login?\nThis information will help me provide the most appropriate solution.",
                    "tone": "professional"
                }
            ],
            "general_inquiry": [
                {
                    "customer": "What are your business hours?",
                    "response": "Our customer service team is available Monday through Friday, 9:00 AM to 6:00 PM EST. For urgent matters outside these hours, you can use our 24/7 automated support system on our website or leave a message, and we'll respond on the next business day.",
                    "tone": "professional"
                },
                {
                    "customer": "Do you ship internationally?",
                    "response": "Yes, we're happy to serve our customers worldwide! We ship to most countries through our trusted shipping partners. Shipping costs and delivery times vary by location. Would you like me to check the specific details for your country?",
                    "tone": "friendly"
                }
            ],
            "billing": [
                {
                    "customer": "I was charged twice for the same order.",
                    "response": "Please don't worry, we'll get this sorted out. Duplicate charges are usually a temporary authorization hold that drops off within 3-5 business days. If you share your order number, I'll confirm whether a second charge actually went through and, if it did, refund it right away.",
                    "tone": "reassuring"
                },
                {
                    "customer": "Can I get an invoice for my purchase?",
                    "response": "Yes. Go to Account > Orders, select the order, and click 'Download Invoice.' If you need it addressed to a company, reply with the company name and I'll send an updated copy.",
                    "tone": "concise"
                }
            ],
            "order_status": [
                {
                    "customer": "Where is my order? It's been a week.",
                    "response": "I'm sorry for the wait, and I understand you'd like your order soon. You can track it anytime under Account > Orders using the tracking link. If you share your order number, I'll check its current location and estimated delivery date for you.",
                    "tone": "empathetic"
                },
                {
                    "customer": "Has my order shipped yet?",
                    "response": "Orders ship within 1-2 business days, and you'll get an email with a tracking number once yours leaves our warehouse. Send me your order number and I'll confirm its status.",
                    "tone": "concise"
                }
            ]
        }

    def load_examples(self, category: str) -> List[Dict]:
        """
        Load relevant examples for the given category
        
        Args:
            category: The type of customer service interaction
            
        Returns:
            List of example interactions
            
        Raises:
            ValueError: If category is invalid
        """
        if category not in self.example_database:
            raise ValueError(f"Invalid category. Must be one of: {list(self.example_database.keys())}")
            
        return self.example_database[category]

    def _create_prompt(self,
                      customer_message: str,
                      category: str,
                      tone: str,
                      examples: List[Dict]) -> str:
        """
        Create prompt with examples and specific instructions
        """
        # Start with tone instruction
        prompt = f"Tone: {tone} ({self.valid_tones[tone]})\n"
        prompt += f"Category: {category.replace('_', ' ')}\n"
        prompt += f"Guidelines: {self.category_guidelines.get(category, 'Address the customer’s request directly and helpfully.')}\n\n"
        prompt += (
            "Rules:\n"
            "- Address the customer's specific issue, not a generic version of it.\n"
            "- Keep the response under 120 words.\n"
            "- Don't invent policies, prices, or order details beyond what the examples show.\n"
            "- End with a clear next step for the customer.\n\n"
        )
        
        # Add examples to enable few-shot learning
        prompt += "Here are examples of high-quality responses:\n\n"
        for i, example in enumerate(examples, 1):
            prompt += f"Example {i} (tone: {example['tone']})\n"
            prompt += f"Customer: {example['customer']}\n"
            prompt += f"Response: {example['response']}\n\n"
        
        # Add the new customer message
        prompt += f"Now write the response for this customer.\nCustomer: {customer_message}\nResponse:"
        return prompt

    def generate_response(self,
                         customer_message: str,
                         category: str,
                         tone: str = "professional",
                         examples: Optional[List[Dict]] = None) -> str:
        """
        Generate a response based on customer message and examples
        """
        # Validate inputs: customer_message, category, tone
        if not isinstance(customer_message, str) or not customer_message.strip():
            raise ValueError("Customer message cannot be empty.")
        if category not in self.example_database:
            raise ValueError(f"Invalid category '{category}'. Must be one of: {list(self.example_database.keys())}")
        if tone not in self.valid_tones:
            raise ValueError(f"Invalid tone '{tone}'. Must be one of: {list(self.valid_tones.keys())}")
            
        # Get examples if not provided
        if examples is None:
            examples = self.load_examples(category)
            
        if not examples:
            raise ValueError(f"No examples available for category: {category}")
        
        # Select the most relevant examples for the tone
        tone_matched_examples = [ex for ex in examples if ex.get('tone') == tone]
        if tone_matched_examples:
            examples = tone_matched_examples[:2]  # Use up to 2 tone-matched examples
        else:
            examples = examples[:2]  # Use first 2 examples if no tone matches
            
        # Create prompt
        prompt = self._create_prompt(customer_message, category, tone, examples)
        
        try:
            # Get response from OpenAI
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an experienced customer service agent who writes clear, accurate, and helpful replies."},
                    {"role": "user", "content": prompt}
                ],
                reasoning_effort=REASONING_EFFORT,
                max_completion_tokens=MAX_COMPLETION_TOKENS
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            logger.error(f"Error generating response: {str(e)}")
            raise

In [ ]:
def main():
    """
    Demonstrate the customer service bot functionality with various scenarios
    """
    # Get API key
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Please set OPENAI_API_KEY environment variable")

    # Initialize bot
    bot = CustomerServiceBot(api_key)

    # Test scenarios
    test_messages = [
        {
            "message": "My product arrived damaged. I want a refund.",
            "category": "refund",
            "tone": "empathetic"
        },
        {
            "message": "How do I reset my password?",
            "category": "technical_support",
            "tone": "professional"
        },
        {
            "message": "What's your shipping policy?",
            "category": "general_inquiry",
            "tone": "friendly"
        },
        {
            "message": "The instructions in your manual are wrong!",
            "category": "technical_support",
            "tone": "empathetic"
        },
        {
            "message": "I think I was billed twice this month.",
            "category": "billing",
            "tone": "reassuring"
        },
        {
            "message": "Has my order shipped?",
            "category": "order_status",
            "tone": "concise"
        }
    ]

    # Process each test message
    for test in test_messages:
        try:
            print("\n" + "="*50)
            print(f"Customer Message: {test['message']}")
            print(f"Category: {test['category']}")
            print(f"Tone: {test['tone']}")
            print(f"Model: {bot.model}")
            print("-"*50)
            
            response = bot.generate_response(
                test['message'],
                test['category'],
                test['tone']
            )
            
            print("Bot Response:")
            print(response)
            print("="*50)
            
        except Exception as e:
            print(f"Error: {str(e)}")

In [ ]:
if __name__ == "__main__":
    main()
    
    print("\nRunning implementation tests...")
    # Create bot instance for testing
    api_key = os.getenv("OPENAI_API_KEY")
    bot = CustomerServiceBot(api_key)
        
    # Create test bench instance
    tester = TestAICustomerServiceBot(bot)
    
    # Run tests and print results
    results = tester.run_all_tests()
    print_test_results(results)